In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import pandas as pd
import numpy as np
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from sklearn.metrics import mean_absolute_error
import joblib

In [17]:
df = pd.read_csv('/content/drive/MyDrive/car-trader/car-trader-dataset/cars-data-cleaned.csv')
df.head()

,car-maker,year,mileage-km,fuel-type,transmission-type,price-rwf
0,toyota,2017,141520,hybrid,automatic,13528556
1,bmw,2016,139091,gasoline,automatic,27587930
2,honda,2007,258093,gasoline,automatic,7376030
3,hyundai,2017,147560,hybrid,automatic,24723848
4,ford,2002,284431,gasoline,automatic,2554924


In [18]:
df_model = df.copy()

In [19]:
# Features
x = df_model.drop('price-rwf', axis=1)

# Target
y = df_model['price-rwf']

In [20]:
# Separate TEST set (20% of all data) # x_temp = 80% # x_test = 20%
x_temp, x_test, y_temp, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

# Step 2: Split that 80% into TRAIN (80% of 80% = 64%) and VAL (20% of 80% = 16%)
x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size=0.2, random_state=42)

In [21]:
# one hot encode
x_train = pd.get_dummies(x_train, columns=['car-maker', 'transmission-type', 'fuel-type'])
x_val = pd.get_dummies(x_val, columns=['car-maker', 'transmission-type', 'fuel-type']).reindex(columns=x_train.columns, fill_value=0)
x_test = pd.get_dummies(x_test, columns=['car-maker', 'transmission-type', 'fuel-type']).reindex(columns=x_train.columns, fill_value=0)

In [22]:
# scale features
features_scaler = StandardScaler()
x_train = features_scaler.fit_transform(x_train)
x_val = features_scaler.transform(x_val)
x_test = features_scaler.transform(x_test)

In [23]:
# Scale target variable
target_scaler = StandardScaler()
y_train = target_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_val = target_scaler.transform(y_val.values.reshape(-1, 1))
y_test = target_scaler.transform(y_test.values.reshape(-1, 1))

In [24]:
class RealMAECallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Predict on validation set
        y_pred_scaled = self.model.predict(x_val, verbose=0)

        # Inverse transform to get real RWF values
        y_pred = target_scaler.inverse_transform(y_pred_scaled)
        y_true = target_scaler.inverse_transform(y_val)

        # Calculate real MAE
        mae = mean_absolute_error(y_true, y_pred)
        print(f"\n[Epoch {epoch+1}] Real Validation MAE: {mae:,.0f} RWF")

In [25]:
# model architecture
model = Sequential([
    Input(shape=(x_train.shape[1],)),
    Dense(512, activation='relu'),
    Dropout(0.1),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(1)
])

#
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

#
early_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

In [26]:
history = model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=128,
    validation_data=(x_val, y_val),
    callbacks=[early_stopping, RealMAECallback()],
    verbose=2
)

Epoch 1/15

[Epoch 1] Real Validation MAE: 5,404,465 RWF
42162/42162 - 385s - 9ms/step - loss: 0.2685 - mae: 0.2846 - val_loss: 0.2668 - val_mae: 0.2792
Epoch 2/15

[Epoch 2] Real Validation MAE: 5,432,500 RWF
42162/42162 - 405s - 10ms/step - loss: 0.2659 - mae: 0.2828 - val_loss: 0.2658 - val_mae: 0.2807
Epoch 3/15

[Epoch 3] Real Validation MAE: 5,466,214 RWF
42162/42162 - 397s - 9ms/step - loss: 0.2654 - mae: 0.2826 - val_loss: 0.2664 - val_mae: 0.2824
Epoch 4/15

[Epoch 4] Real Validation MAE: 5,442,316 RWF
42162/42162 - 407s - 10ms/step - loss: 0.2651 - mae: 0.2823 - val_loss: 0.2656 - val_mae: 0.2812
Epoch 5/15

[Epoch 5] Real Validation MAE: 5,488,638 RWF
42162/42162 - 394s - 9ms/step - loss: 0.2650 - mae: 0.2825 - val_loss: 0.2674 - val_mae: 0.2836
Epoch 6/15

[Epoch 6] Real Validation MAE: 5,565,210 RWF
42162/42162 - 362s - 9ms/step - loss: 0.2649 - mae: 0.2826 - val_loss: 0.2665 - val_mae: 0.2875
Epoch 7/15

[Epoch 7] Real Validation MAE: 5,449,718 RWF
42162/42162 - 369s - 9m

In [27]:
# 1. Make predictions
y_pred = model.predict(x_test)

# 2. Inverse transform to actual prices
y_pred_actual = target_scaler.inverse_transform(y_pred)
y_test_actual = target_scaler.inverse_transform(y_test)

comparison = pd.DataFrame({
    'Predicted Price (rwf)': y_pred_actual.flatten(),
    'Actual Price (rwf)': y_test_actual.flatten(),
    'Difference (rwf)': y_test_actual.flatten() - y_pred_actual.flatten()
})

print(comparison.head(10))

52702/52702 ━━━━━━━━━━━━━━━━━━━━ 72s 1ms/step
   Predicted Price (rwf)  Actual Price (rwf)  Difference (rwf)
0             1627086.25           2307014.0         679927.75
1            48597188.00          79676590.0       31079402.00
2            20214716.00          22688075.0        2473359.00
3             4061795.25           1929317.0       -2132478.25
4             3020745.25           3454689.0         433943.75
5             1095296.25           1166632.0          71335.75
6            15199058.00           9801167.0       -5397891.00
7             2266172.25           2893247.0         627074.75
8             1312585.25           1166632.0        -145953.25
9            36184692.00          30797626.0       -5387066.00


In [28]:
# Save the model
model.save('/content/drive/MyDrive/car-trader/car-trader-models-ann/car-trader-ann.keras')

# Save the scalers (you need these for predictions!)
joblib.dump(features_scaler, '/content/drive/MyDrive/car-trader/car-trader-models-ann/features-scaler-ann.pkl')
joblib.dump(target_scaler, '/content/drive/MyDrive/car-trader/car-trader-models-ann/target-scaler-ann.pkl')

['/content/drive/MyDrive/car-trader/car-trader-models-ann/target-scaler-ann.pkl']